# 🏥 Day 3 — Healthcare OOP Assignment

## EXL AI & Data Engineering Training — Intermediate → Advanced

---

> **This is your graded assignment. Submit via GitHub before the session ends.**
> File: `day3_assignment.ipynb` + `reports/` folder with all CSV outputs.

---

### 🎯 Assignment Overview

You are a **Data Engineering consultant** hired by **EXL Healthcare Analytics**.
They have three years of raw patient data across multiple hospitals.
Your job: build a **production-grade OOP data system** from scratch using the patterns from today.

### 📊 Scoring Rubric

| Section   | Marks   | Topic                               |
| --------- | ------- | ----------------------------------- |
| Section A | 15      | Data Models with @dataclass         |
| Section B | 20      | OOP Pipeline with ABC + Inheritance |
| Section C | 20      | Factory + Singleton Patterns        |
| Section D | 25      | Advanced Analysis & Transforms      |
| Section E | 20      | Integration — Full System           |
| **Total** | **100** |                                     |

### ⚠️ Rules

- Every `# YOUR CODE HERE` block must be replaced with working code
- All outputs must print correctly when cells are run top-to-bottom
- Do NOT delete any existing cells or assertion blocks
- You may add extra helper cells — label them `# SCRATCH`

---

### Setup — Run this first


In [4]:
import pandas as pd
import numpy as np
import json, os, re, csv
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional, ClassVar, Type, Iterator
from datetime import datetime, date
import warnings
warnings.filterwarnings("ignore")

# ── Create realistic healthcare dataset ─────────────────────────
np.random.seed(42)
N = 500

DIAGNOSES = ["Diabetes Type 2","Hypertension","Cardiac Arrhythmia",
             "COPD","Chronic Kidney Disease","Obesity","Depression",
             "Asthma","Osteoporosis","Anaemia"]
DEPARTMENTS = ["Cardiology","Endocrinology","Nephrology","Pulmonology",
               "Psychiatry","General Medicine","Orthopaedics","Neurology"]
WARDS = ["Ward-A","Ward-B","ICU","HDU","Day-Care","Emergency"]
GENDERS = ["Male","Female","Non-Binary"]

raw = {
    "patient_id":   [f"EXL{str(i).zfill(4)}" for i in range(1, N+1)],
    "name":         [f"Patient {chr(65+i%26)}{chr(65+(i//26)%26)}" for i in range(N)],
    "age":          np.random.randint(18, 88, N).tolist(),
    "gender":       np.random.choice(GENDERS, N).tolist(),
    "diagnosis":    np.random.choice(DIAGNOSES, N).tolist(),
    "department":   np.random.choice(DEPARTMENTS, N).tolist(),
    "ward":         np.random.choice(WARDS, N).tolist(),
    "admission_score": np.round(np.random.uniform(20, 100, N), 1).tolist(),
    "discharge_score": np.round(np.random.uniform(40, 100, N), 1).tolist(),
    "los_days":     np.random.randint(1, 45, N).tolist(),   # length of stay
    "readmitted":   np.random.choice([True, False], N, p=[0.22, 0.78]).tolist(),
    "cost_usd":     np.round(np.random.uniform(500, 85000, N), 2).tolist(),
    "bmi":          np.round(np.random.uniform(16.0, 45.0, N), 1).tolist(),
    "systolic_bp":  np.random.randint(90, 185, N).tolist(),
    "diastolic_bp": np.random.randint(60, 120, N).tolist(),
    "heart_rate":   np.random.randint(45, 135, N).tolist(),
    "hba1c":        np.round(np.random.uniform(4.5, 12.5, N), 1).tolist(),
}

# Inject realistic nulls (5–12% per column)
df_raw = pd.DataFrame(raw)
for col in ["bmi","hba1c","systolic_bp","diastolic_bp","heart_rate"]:
    mask = np.random.random(N) < 0.08
    df_raw.loc[mask, col] = np.nan
for col in ["cost_usd","los_days"]:
    mask = np.random.random(N) < 0.05
    df_raw.loc[mask, col] = np.nan

# Inject a few dirty records
df_raw.loc[0,  "age"]    = -5        # invalid
df_raw.loc[1,  "bmi"]    = 200.0     # invalid
df_raw.loc[2,  "name"]   = "  patient bb  "  # needs stripping
df_raw.loc[3,  "gender"] = "MALE"    # inconsistent case
df_raw.loc[4,  "hba1c"]  = 0.1      # physiologically impossible

os.makedirs("reports", exist_ok=True)
df_raw.to_csv("healthcare_raw.csv", index=False)
print(f"✅ Dataset created: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"   Columns: {list(df_raw.columns)}")
print(f"   Nulls:   {df_raw.isnull().sum().sum()} total")
print(f"   Saved:   healthcare_raw.csv")

✅ Dataset created: 500 rows × 17 columns
   Columns: ['patient_id', 'name', 'age', 'gender', 'diagnosis', 'department', 'ward', 'admission_score', 'discharge_score', 'los_days', 'readmitted', 'cost_usd', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'hba1c']
   Nulls:   258 total
   Saved:   healthcare_raw.csv


---

## Section A — Data Models with `@dataclass` (15 marks)

Build the data layer that will underpin the entire assignment.
Every other section uses these models.

### A1 (5 marks) — `Vitals` dataclass

### A2 (5 marks) — `PatientRecord` dataclass

### A3 (5 marks) — `HospitalBed` dataclass with occupancy logic


In [ ]:
from dataclasses import dataclass
from typing import Optional

# ── A1 (5 marks): Vitals dataclass ─────────────────────────────
# Requirements:
#   Fields: systolic_bp (int), diastolic_bp (int),
#           heart_rate (int), hba1c (float), bmi (float)
#   All fields have Optional type with default None
#   __post_init__: validate each field only when not None:
#       systolic_bp:  60 ≤ x ≤ 250
#       diastolic_bp: 40 ≤ x ≤ 160
#       heart_rate:   30 ≤ x ≤ 200
#       hba1c:        3.0 ≤ x ≤ 20.0
#       bmi:          10.0 ≤ x ≤ 80.0
#   Computed property (not stored): bp_category
#       systolic >= 140 or diastolic >= 90 → "Hypertensive"
#       systolic >= 120 or diastolic >= 80 → "Elevated"
#       else → "Normal"
#   Computed property: metabolic_risk_score (float, 0–100)
#       Formula:
#           base = 0
#           if hba1c is not None and hba1c > 7.0:  base += 25
#           if bmi    is not None and bmi > 30.0:  base += 20
#           if bp_category == "Hypertensive":       base += 30
#           if bp_category == "Elevated":           base += 15
#           return min(base, 100)
#   Method: to_dict() → dict of non-None values only

@dataclass
class Vitals:
    systolic_bp: Optional[int] = None
    diastolic_bp: Optional[int] = None
    heart_rate: Optional[int] = None
    hba1c: Optional[float] = None
    bmi: Optional[float] = None

    def __post_init__(self):
        if self.systolic_bp is not None and not (60 <= self.systolic_bp <= 250):
            raise ValueError(f"systolic_bp out of range: {self.systolic_bp}")
        if self.diastolic_bp is not None and not (40 <= self.diastolic_bp <= 160):
            raise ValueError(f"diastolic_bp out of range: {self.diastolic_bp}")
        if self.heart_rate is not None and not (30 <= self.heart_rate <= 200):
            raise ValueError(f"heart_rate out of range: {self.heart_rate}")
        if self.hba1c is not None and not (3.0 <= self.hba1c <= 20.0):
            raise ValueError(f"hba1c out of range: {self.hba1c}")
        if self.bmi is not None and not (10.0 <= self.bmi <= 80.0):
            raise ValueError(f"bmi out of range: {self.bmi}")

    @property
    def bp_category(self) -> str:
        s = self.systolic_bp
        d = self.diastolic_bp
        if (s is not None and s >= 140) or (d is not None and d >= 90):
            return "Hypertensive"
        if (s is not None and s >= 120) or (d is not None and d >= 80):
            return "Elevated"
        return "Normal"

    @property
    def metabolic_risk_score(self) -> float:
        base = 0
        if self.hba1c is not None and self.hba1c > 7.0:
            base += 25
        if self.bmi is not None and self.bmi > 30.0:
            base += 20
        if self.bp_category == "Hypertensive":
            base += 30
        elif self.bp_category == "Elevated":
            base += 15
        return float(min(base, 100))

    def to_dict(self) -> dict:
        return {k: v for k, v in self.__dict__.items() if v is not None}
    
# ── Tests — do not modify ────────────────────────────────────────
try:
    v1 = Vitals(systolic_bp=145, diastolic_bp=95, heart_rate=88,
                hba1c=8.2, bmi=31.5)
    assert v1.bp_category == "Hypertensive", f"Expected Hypertensive, got {v1.bp_category}"
    assert v1.metabolic_risk_score == 75, f"Expected 75, got {v1.metabolic_risk_score}"
    assert "bmi" in v1.to_dict(), "to_dict() missing bmi"

    v2 = Vitals(systolic_bp=118, diastolic_bp=75, heart_rate=72)
    assert v2.bp_category == "Normal", f"Expected Normal, got {v2.bp_category}"
    assert v2.metabolic_risk_score == 0, f"Expected 0, got {v2.metabolic_risk_score}"
    assert "hba1c" not in v2.to_dict(), "to_dict() should skip None hba1c"

    try:
        Vitals(systolic_bp=300)  # should raise ValueError
        print("❌ Validation not working for systolic_bp=300")
    except ValueError:
        pass

    print("✅ A1 Vitals — all tests passed (5/5 marks)")
except Exception as e:
    print(f"❌ A1 failed: {e}")


✅ A1 Vitals — all tests passed (5/5 marks)


In [30]:
# ── A2 (5 marks): PatientRecord dataclass ───────────────────────
# Requirements:
#   Fields: patient_id (str), name (str), age (int),
#           gender (str), diagnosis (str), department (str),
#           ward (str), admission_score (float),
#           discharge_score (float) = 0.0,
#           los_days (int) = 0,
#           readmitted (bool) = False,
#           cost_usd (float) = 0.0,
#           vitals (Optional[Vitals]) = None,
#           tags (list[str]) = field(default_factory=list)
#   ClassVar: VALID_GENDERS = {"Male","Female","Non-Binary"}
#   __post_init__:
#       - Validate: 0 < age < 120
#       - Validate: gender in VALID_GENDERS (raise ValueError if not)
#       - Auto-clean: name.strip().title()
#       - Auto-clean: gender.strip().title()
#       - Compute: self.score_improvement = discharge_score - admission_score
#   Properties:
#       risk_tier (str): "Critical" if admission_score < 40
#                        "High"     if admission_score < 60
#                        "Moderate" if admission_score < 80
#                        else "Stable"
#       recovery_rate (float): score_improvement / los_days if los_days > 0 else 0.0
#   Methods:
#       add_tag(tag: str) → self
#       to_dict() → full dict including risk_tier, recovery_rate, vitals as dict

@dataclass
class PatientRecord:
    # YOUR CODE HERE
    patient_id: str
    name: str
    age: int
    gender: str
    diagnosis: str
    department: str
    ward: str
    admission_score: float
    discharge_score: float = 0.0
    los_days: int = 0
    readmitted: bool = False
    cost_usd: float = 0.0
    vitals: Optional[Vitals] = None
    tags: list[str] = field(default_factory=list)
    VALID_GENDERS: ClassVar[set] = {"Male", "Female", "Non-Binary"}

    def __post_init__(self):
        if not (0 < self.age < 120):
            raise ValueError(f"age out of range: {self.age}")
        
        self.name = self.name.strip().title()
        self.gender = self.gender.strip().title()
        
        if self.gender not in self.VALID_GENDERS:
            raise ValueError(f"gender '{self.gender}' not in {self.VALID_GENDERS}")
        
        self.score_improvement = self.discharge_score - self.admission_score
    @property
    def risk_tier(self) -> str:
        if self.admission_score < 40:
            return "Critical"
        elif self.admission_score < 60:
            return "High"
        elif self.admission_score < 80:
            return "Moderate"
        else:
            return "Stable"
    @property
    def recovery_rate(self) -> float:
        if self.los_days > 0:
            return self.score_improvement / self.los_days
        return 0.0
    def add_tag(self, tag: str) -> 'PatientRecord':
        self.tags.append(tag)
        return self
    def to_dict(self) -> dict:
        base_dict = {
            "patient_id": self.patient_id,
            "name": self.name,
            "age": self.age,
            "gender": self.gender
        }
        base_dict.update({
            "diagnosis": self.diagnosis,
            "department": self.department,
            "ward": self.ward,
            "admission_score": self.admission_score,
            "discharge_score": self.discharge_score,
            "los_days": self.los_days,
            "readmitted": self.readmitted,
            "cost_usd": self.cost_usd,
            "risk_tier": self.risk_tier,
            "recovery_rate": self.recovery_rate,
            "tags": self.tags
        })
        if self.vitals is not None:
            base_dict["vitals"] = self.vitals.to_dict()
        return base_dict

# ── Tests — do not modify ────────────────────────────────────────
try:
    pr = PatientRecord(
        patient_id="EXL0001", name="  alice singh  ", age=52,
        gender="female", diagnosis="Diabetes Type 2",
        department="Endocrinology", ward="Ward-A",
        admission_score=55.0, discharge_score=78.0, los_days=8, cost_usd=4200.0
    )
    assert pr.name == "Alice Singh",       f"name not cleaned: {pr.name}"
    assert pr.gender == "Female",          f"gender not cleaned: {pr.gender}"
    assert pr.risk_tier == "High",         f"Expected High, got {pr.risk_tier}"
    assert abs(pr.score_improvement - 23.0) < 0.01
    assert abs(pr.recovery_rate - 23/8) < 0.01
    pr.add_tag("follow-up").add_tag("high-cost")
    assert len(pr.tags) == 2
    d = pr.to_dict()
    assert "risk_tier" in d and "recovery_rate" in d

    try:
        PatientRecord("X","Y",150,"Male","D","Dep","W",50.0)  # age 150 → invalid
        print("❌ age validation missing")
    except (ValueError, AssertionError):
        pass

    try:
        PatientRecord("X","Y",30,"Unknown","D","Dep","W",50.0)  # bad gender
        print("❌ gender validation missing")
    except ValueError:
        pass

    print("✅ A2 PatientRecord — all tests passed (5/5 marks)")
except Exception as e:
    print(f"❌ A2 failed: {e}")

✅ A2 PatientRecord — all tests passed (5/5 marks)


In [ ]:
# ── A3 (5 marks): HospitalBed dataclass ────────────────────────
# Requirements:
#   Fields: bed_id (str), ward (str), capacity (int),
#           current_patients (list[str]) = field(default_factory=list)
#   ClassVar: total_beds_created: int = 0 (increment in __post_init__)
#   __post_init__: validate capacity > 0, increment class counter
#   Properties:
#       occupancy_rate (float): len(current_patients) / capacity
#       is_full (bool): occupancy_rate >= 1.0
#       available_slots (int): capacity - len(current_patients)
#   Methods:
#       admit(patient_id: str) → self
#           Raise RuntimeError if bed is full
#           Raise ValueError if patient_id already admitted
#       discharge(patient_id: str) → self
#           Raise ValueError if patient_id not found
#       status_report() → dict  with bed_id, ward, capacity,
#           occupied, available, occupancy_pct (rounded to 1dp)

@dataclass
class HospitalBed:
    bed_id: str
    ward: str
    capacity: int
    current_patients: list[str] = field(default_factory=list)
    total_beds_created: ClassVar[int] = 0

    def __post_init__(self):
        if self.capacity <= 0:
            raise ValueError(f"capacity must be > 0, got {self.capacity}")
        HospitalBed.total_beds_created += 1

    @property
    def occupancy_rate(self) -> float:
        return len(self.current_patients) / self.capacity

    @property
    def is_full(self) -> bool:
        return self.occupancy_rate >= 1.0

    @property
    def available_slots(self) -> int:
        return self.capacity - len(self.current_patients)

    def admit(self, patient_id: str) -> 'HospitalBed':
        if patient_id in self.current_patients:
            raise ValueError(f"Patient {patient_id} already admitted in bed {self.bed_id}")
        if self.is_full:
            raise RuntimeError(f"Cannot admit {patient_id}: bed {self.bed_id} is full")
        self.current_patients.append(patient_id)
        return self

    def discharge(self, patient_id: str) -> 'HospitalBed':
        if patient_id not in self.current_patients:
            raise ValueError(f"Patient {patient_id} not found in bed {self.bed_id}")
        self.current_patients.remove(patient_id)
        return self

    def status_report(self) -> dict:
        occupied = len(self.current_patients)
        available = self.available_slots
        occupancy_pct = round((occupied / self.capacity) * 100, 1)
        return {
            "bed_id": self.bed_id,
            "ward": self.ward,
            "capacity": self.capacity,
            "occupied": occupied,
            "available": available,
            "occupancy_pct": occupancy_pct,
        }
    
# ── Tests — do not modify ────────────────────────────────────────
try:
    b1 = HospitalBed("BED-001", "ICU", capacity=4)
    b2 = HospitalBed("BED-002", "Ward-A", capacity=6)
    assert HospitalBed.total_beds_created == 2

    b1.admit("EXL0001").admit("EXL0002").admit("EXL0003")
    assert b1.occupancy_rate == 0.75
    assert b1.available_slots == 1
    assert not b1.is_full

    b1.admit("EXL0004")
    assert b1.is_full

    try:
        b1.admit("EXL0005")   # full — should raise RuntimeError
        print("❌ Full bed should raise RuntimeError")
    except RuntimeError:
        pass

    try:
        b1.admit("EXL0001")   # duplicate — should raise ValueError
        print("❌ Duplicate admit should raise ValueError")
    except ValueError:
        pass

    b1.discharge("EXL0002")
    assert b1.available_slots == 1
    rpt = b1.status_report()
    assert rpt["occupancy_pct"] == 75.0

    print(f"✅ A3 HospitalBed — all tests passed (5/5 marks)")
    print(f"   Total beds created: {HospitalBed.total_beds_created}")
except Exception as e:
    print(f"❌ A3 failed: {e}")


✅ A3 HospitalBed — all tests passed (5/5 marks)
   Total beds created: 2


---

## Section B — OOP Pipeline with ABC + Inheritance (20 marks)

Build the data ingestion and cleaning hierarchy.

### B1 (6 marks) — `HealthcareDataLoader` ABC

### B2 (7 marks) — `CSVHealthcareLoader` concrete class

### B3 (7 marks) — `HealthcarePipeline` with full method chaining


In [36]:
# ── B1 (6 marks): HealthcareDataLoader ABC ──────────────────────
# Requirements:
#   Abstract methods: load(path) → pd.DataFrame
#                     validate(df) → bool
#                     get_schema() → dict  (column → expected dtype)
#   Concrete methods:
#       preview(df, rows=5):
#           Print loader class name, shape, null counts, first N rows
#       profile(df) → dict:
#           Returns: {
#             "rows": int, "cols": int,
#             "null_pct": float (rounded to 2dp),
#             "numeric_cols": list[str],
#             "categorical_cols": list[str],
#             "duplicates": int
#           }
#       schema_check(df) → list[str]:
#           Compares df.dtypes against get_schema().
#           Returns list of warnings like:
#           "Column 'age' expected int64, found object"
#           Returns [] if all types match

class HealthcareDataLoader(ABC):
    @abstractmethod
    def load(self, path: str) -> pd.DataFrame:
        pass
    @abstractmethod
    def validate(self, df: pd.DataFrame) -> bool:
        pass
    @abstractmethod
    def get_schema(self) -> dict:
        pass
    def preview(self, df: pd.DataFrame, rows: int = 5):
        print(f"Loader: {self.__class__.__name__}")
        print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"Null counts:\n{df.isnull().sum()}")
        print(f"First {rows} rows:")
        print(df.head(rows))
    def profile(self, df: pd.DataFrame) -> dict:
        total_cells = df.size
        null_cells = df.isnull().sum().sum()
        null_pct = round((null_cells / total_cells) * 100, 2)
        numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
        categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
        duplicates = df.duplicated().sum()
        return {
            "rows": df.shape[0],
            "cols": df.shape[1],
            "null_pct": null_pct,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
            "duplicates": duplicates
        }
    def schema_check(self, df: pd.DataFrame) -> list[str]:
        expected_schema = self.get_schema()
        warnings = []
        for col, expected_dtype in expected_schema.items():
            if col in df.columns:
                actual_dtype = df[col].dtype
                if str(actual_dtype) != expected_dtype:
                    warnings.append(f"Column '{col}' expected {expected_dtype}, found {actual_dtype}")
            else:
                warnings.append(f"Column '{col}' missing from DataFrame")
        return warnings

# ── Tests ────────────────────────────────────────────────────────
try:
    # Can't instantiate ABC
    try:
        HealthcareDataLoader()
        print("❌ ABC should not be instantiable")
    except TypeError:
        pass
    print("✅ B1 HealthcareDataLoader ABC — structure correct (6/6 marks)")
except Exception as e:
    print(f"❌ B1 failed: {e}")

✅ B1 HealthcareDataLoader ABC — structure correct (6/6 marks)


In [9]:
# ── B2 (7 marks): CSVHealthcareLoader ──────────────────────────
# Inherits: HealthcareDataLoader
# Requirements:
#   __init__(encoding="utf-8", parse_dates=None)
#       parse_dates: list of column names to parse as datetime
#   load(path):
#       pd.read_csv with encoding, parse_dates
#       Wrap FileNotFoundError and pd.errors.ParserError
#       Print: f"Loaded {rows} rows × {cols} cols from '{path}'"
#   validate(df):
#       REQUIRED = ["patient_id","age","diagnosis","admission_score"]
#       Check all REQUIRED cols exist (raise ValueError listing missing)
#       Check no duplicate patient_ids (raise ValueError with count)
#       Check age column: all values between 0-120 when not null
#           Print warning (don't raise) for rows with bad ages
#       Return True if all hard checks pass
#   get_schema() → dict:
#       {"patient_id":"object","age":"int64",
#        "admission_score":"float64","discharge_score":"float64",
#        "los_days":"float64","cost_usd":"float64",
#        "readmitted":"bool","bmi":"float64",
#        "hba1c":"float64","systolic_bp":"float64"}
#   clean(df) → pd.DataFrame:
#       1. drop_duplicates(subset=["patient_id"], keep="first")
#       2. Fix invalid ages: replace < 0 or > 120 with NaN
#       3. Fix invalid bmi: replace < 10 or > 80 with NaN
#       4. Fix invalid hba1c: replace < 3.0 or > 20.0 with NaN
#       5. For numeric cols: fillna with column MEDIAN
#       6. name: .str.strip().str.title()
#       7. gender: .str.strip().str.title()
#       8. reset_index(drop=True)
#       9. Print: f"Cleaned: {before} → {after} rows"
#       10. Return cleaned df

class CSVHealthcareLoader(HealthcareDataLoader):
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    loader = CSVHealthcareLoader()
    df = loader.load("healthcare_raw.csv")
    assert df.shape[0] == 500, f"Expected 500 rows, got {df.shape[0]}"

    profile = loader.profile(df)
    assert "null_pct" in profile
    assert "numeric_cols" in profile
    print(f"   Profile: {profile['rows']} rows, {profile['null_pct']}% nulls")

    df_clean = loader.clean(df)
    assert df_clean["age"].max() <= 120,  "Invalid ages not fixed"
    assert df_clean["bmi"].min() >= 10.0, "Invalid BMI not fixed"
    assert df_clean.isnull().sum().sum() == 0, "Nulls remain after clean"
    assert df_clean["name"].str[0].str.isupper().all(), "Names not title-cased"

    schema_warnings = loader.schema_check(df_clean)
    print(f"   Schema warnings: {len(schema_warnings)}")

    loader.preview(df_clean)
    print("✅ B2 CSVHealthcareLoader — all tests passed (7/7 marks)")
except Exception as e:
    print(f"❌ B2 failed: {e}")
    import traceback; traceback.print_exc()

❌ B2 failed: 'CSVHealthcareLoader' object has no attribute 'load'


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\4029240786.py", line 42, in <module>
    df = loader.load("healthcare_raw.csv")
         ^^^^^^^^^^^
AttributeError: 'CSVHealthcareLoader' object has no attribute 'load'


In [10]:
# ── B3 (7 marks): HealthcarePipeline ────────────────────────────
# Requirements:
#   __init__(loader: HealthcareDataLoader):
#       Store loader. Initialise: df=None, _loaded=False,
#       _cleaned=False, _transformed=False, _steps=[]
#   load(path) → self:
#       Call loader.load() + loader.validate() + loader.clean()
#       Store clean df. Append "load" to _steps. Set _loaded=True.
#   transform() → self:
#       Guard: RuntimeError if not _loaded
#       Add columns:
#         age_group:     pd.cut [0,30,50,65,120] labels ["Youth","Adult","Middle","Senior"]
#         risk_tier:     "Critical"<40, "High"<60, "Moderate"<80, else "Stable"
#                        based on admission_score
#         score_delta:   discharge_score - admission_score
#         score_zscore:  (admission_score - mean) / std  using numpy
#         cost_per_day:  cost_usd / los_days (fillna 0)
#         bp_status:     "Hypertensive" if systolic_bp>=140 else
#                        "Elevated"     if systolic_bp>=120 else "Normal"
#         high_risk_flag: True if risk_tier in ("Critical","High")
#         readmission_cost: cost_usd * 1.35 if readmitted else cost_usd
#       Append "transform" to _steps. Set _transformed=True.
#   filter(diagnosis=None, risk_tier=None, ward=None,
#           min_age=None, max_age=None) → self:
#       Apply any non-None filter as AND condition.
#       Print: f"Filter applied: {before} → {after} rows"
#       Append "filter" to _steps.
#   export(path, include_index=False) → None:
#       Guard: RuntimeError if not _loaded
#       df.to_csv(path, index=include_index)
#       Print: f"Exported {rows} rows × {cols} cols → '{path}'"
#   summary() → dict:
#       Return pipeline stats — only include keys that exist in df:
#       rows, columns, steps_executed, risk_distribution (value_counts dict),
#       avg_admission_score, avg_discharge_score, readmission_rate,
#       total_cost_usd, avg_los_days

class HealthcarePipeline(ABC if False else object):
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    pipeline = HealthcarePipeline(CSVHealthcareLoader())
    pipeline.load("healthcare_raw.csv").transform()

    assert "age_group" in pipeline.df.columns
    assert "risk_tier" in pipeline.df.columns
    assert "score_delta" in pipeline.df.columns
    assert "bp_status" in pipeline.df.columns
    assert "high_risk_flag" in pipeline.df.columns

    n_before = len(pipeline.df)
    pipeline.filter(risk_tier="High")
    assert len(pipeline.df) < n_before, "Filter had no effect"

    pipeline.export("reports/pipeline_output.csv")
    assert os.path.exists("reports/pipeline_output.csv")

    summary = pipeline.summary()
    assert "rows" in summary
    assert "readmission_rate" in summary
    print(f"   Summary: {json.dumps({k:v for k,v in summary.items() if k in ['rows','readmission_rate','avg_admission_score']}, indent=2)}")
    print("✅ B3 HealthcarePipeline — all tests passed (7/7 marks)")
except Exception as e:
    print(f"❌ B3 failed: {e}")
    import traceback; traceback.print_exc()

❌ B3 failed: HealthcarePipeline() takes no arguments


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\2019527457.py", line 44, in <module>
    pipeline = HealthcarePipeline(CSVHealthcareLoader())
TypeError: HealthcarePipeline() takes no arguments


---

## Section C — Factory + Singleton Patterns (20 marks)

### C1 (8 marks) — `AnalysisFactory` — pluggable analytics engine

### C2 (6 marks) — `HospitalConfig` Singleton — global pipeline settings

### C3 (6 marks) — `ReportCache` Singleton — memoised report results


In [11]:
# ── C1 (8 marks): AnalysisFactory ──────────────────────────────
# Build an analysis engine using the Factory Pattern.
#
# First, define an ABC: HealthcareAnalyser
#   Abstract methods:
#       analyse(df: pd.DataFrame) → dict
#       report_name() → str
#   Concrete method:
#       run(df) → dict:
#           Calls analyse(df), adds metadata:
#           "analyser": self.__class__.__name__
#           "report_name": self.report_name()
#           "generated_at": datetime.now().isoformat()
#           "row_count": len(df)
#           Returns the merged dict
#
# Then implement THREE concrete analysers:
#
# 1. DiagnosisAnalyser:
#       report_name() → "Diagnosis Distribution Report"
#       analyse(df) → {
#           "top_5_diagnoses": value_counts().head(5).to_dict(),
#           "avg_score_by_diagnosis": groupby mean admission_score (rounded 2dp),
#           "readmission_by_diagnosis": groupby readmitted mean (rounded 3dp),
#           "total_cost_by_diagnosis": groupby cost_usd sum (rounded 2dp)
#       }
#
# 2. DemographicsAnalyser:
#       report_name() → "Demographics Report"
#       analyse(df) → {
#           "age_band_distribution": age_group value_counts().to_dict(),
#           "gender_split": gender value_counts(normalize=True).round(3).to_dict(),
#           "avg_age_by_diagnosis": groupby age mean round(1),
#           "avg_bmi_by_ward": groupby ward bmi mean round(2)
#       }
#
# 3. ClinicalRiskAnalyser:
#       report_name() → "Clinical Risk Assessment Report"
#       analyse(df) → {
#           "risk_tier_counts": risk_tier value_counts().to_dict(),
#           "high_risk_readmission_rate": float, mean of readmitted
#               where high_risk_flag==True, rounded to 3dp
#           "critical_patients": list of patient_id where risk_tier=="Critical"
#           "avg_los_by_risk": groupby risk_tier los_days mean round(1),
#           "avg_cost_by_risk": groupby risk_tier cost_usd mean round(2)
#       }
#
# Then build AnalysisFactory:
#   _registry = {"diagnosis": DiagnosisAnalyser,
#                "demographics": DemographicsAnalyser,
#                "clinical_risk": ClinicalRiskAnalyser}
#   create(name, **kwargs) → HealthcareAnalyser instance
#   register(name, cls) — validates subclass of HealthcareAnalyser
#   run_all(df) → dict[str, dict]  runs every registered analyser
#   available() → list[str]

# YOUR CODE HERE

# ── Tests ────────────────────────────────────────────────────────
try:
    # Need transformed pipeline df
    pipe_c = HealthcarePipeline(CSVHealthcareLoader())
    pipe_c.load("healthcare_raw.csv").transform()
    df_c = pipe_c.df.copy()

    factory = AnalysisFactory()
    assert sorted(factory.available()) == ["clinical_risk","demographics","diagnosis"]

    diag = factory.create("diagnosis")
    result = diag.run(df_c)
    assert "top_5_diagnoses" in result
    assert "analyser" in result
    assert "generated_at" in result
    print(f"   Diagnosis report keys: {[k for k in result if k not in ('analyser','report_name','generated_at','row_count')]}")

    demo = factory.create("demographics")
    dresult = demo.run(df_c)
    assert "gender_split" in dresult

    risk = factory.create("clinical_risk")
    rresult = risk.run(df_c)
    assert "risk_tier_counts" in rresult
    assert "critical_patients" in rresult

    all_results = factory.run_all(df_c)
    assert len(all_results) == 3
    print(f"   run_all() produced {len(all_results)} reports")
    print("✅ C1 AnalysisFactory — all tests passed (8/8 marks)")
except Exception as e:
    print(f"❌ C1 failed: {e}")
    import traceback; traceback.print_exc()

❌ C1 failed: HealthcarePipeline() takes no arguments


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\2082269292.py", line 62, in <module>
    pipe_c = HealthcarePipeline(CSVHealthcareLoader())
TypeError: HealthcarePipeline() takes no arguments


In [12]:
# ── C2 (6 marks): HospitalConfig Singleton ──────────────────────
# Build a thread-safe Singleton for global pipeline configuration.
#
# Requirements:
#   Singleton mechanics: __new__ + _initialised guard
#   Fields (set in __init__ on first call):
#       _config: dict — starts empty
#       _history: list[dict] — tracks all set() calls with timestamp
#   Methods:
#       set(key, value) → self   — store + record to history
#       get(key, default=None)   — retrieve
#       load_defaults() → self   — sets:
#           "score_threshold":    60.0
#           "high_risk_threshold": 40.0
#           "max_los_days":       30
#           "readmission_window": 30
#           "cost_outlier_pct":   0.95
#           "hospital_name":      "EXL General Hospital"
#           "fiscal_year":        2025
#       update(d: dict) → self   — bulk update from dict
#       history() → list[dict]   — return copy of _history
#       snapshot() → dict        — return copy of _config
#       reset() → self           — clear config + history (for testing)
#   __repr__: show hospital_name and key count

class HospitalConfig:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    # Reset first (in case previous tests ran)
    cfg1 = HospitalConfig()
    cfg1.reset()

    cfg1 = HospitalConfig()
    cfg2 = HospitalConfig()
    assert cfg1 is cfg2, "Not a singleton!"

    cfg1.load_defaults()
    assert cfg2.get("score_threshold") == 60.0
    assert cfg2.get("hospital_name")   == "EXL General Hospital"

    cfg1.set("score_threshold", 65.0)
    assert cfg2.get("score_threshold") == 65.0   # same object!

    hist = cfg1.history()
    assert len(hist) >= 7, f"Expected ≥7 history entries, got {len(hist)}"
    assert all("timestamp" in h for h in hist)

    snap = cfg2.snapshot()
    assert "hospital_name" in snap
    print(f"   Config: {cfg2}")
    print(f"   Keys: {sorted(snap.keys())}")
    print("✅ C2 HospitalConfig — all tests passed (6/6 marks)")
except Exception as e:
    print(f"❌ C2 failed: {e}")

❌ C2 failed: 'HospitalConfig' object has no attribute 'reset'


In [13]:
# ── C3 (6 marks): ReportCache Singleton ─────────────────────────
# A memoisation cache for analysis results.
# Once a report is generated, it's cached so repeated calls are instant.
#
# Requirements:
#   Singleton mechanics (same pattern as HospitalConfig)
#   Fields: _cache: dict[str, dict], _hit_count: int, _miss_count: int
#   Methods:
#       get(key) → dict | None
#           Increment _hit_count if found, else _miss_count
#       set(key, value: dict) → self
#           value must be a dict (raise TypeError otherwise)
#       invalidate(key) → bool   returns True if key existed
#       clear() → self           empties cache, resets counters
#       stats() → dict:
#           {"size": int, "hits": int, "misses": int,
#            "hit_rate": float (0.0 if no requests)}
#       keys() → list[str]
#       get_or_compute(key, compute_fn, df) → dict:
#           If key in cache: return cached value (hit)
#           Else: call compute_fn(df), cache result, return it (miss)
#           Print: "CACHE HIT: key" or "CACHE MISS: key — computed"

class ReportCache:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    rc1 = ReportCache(); rc1.clear()
    rc2 = ReportCache()
    assert rc1 is rc2

    rc1.set("test_report", {"data": [1,2,3], "rows": 3})
    assert rc2.get("test_report") == {"data":[1,2,3],"rows":3}

    rc1.get("nonexistent")    # miss
    stats = rc1.stats()
    assert stats["hits"]   == 1
    assert stats["misses"] == 1
    assert stats["hit_rate"] == 0.5

    # get_or_compute test
    pipe_c2 = HealthcarePipeline(CSVHealthcareLoader())
    pipe_c2.load("healthcare_raw.csv").transform()
    df_c2 = pipe_c2.df.copy()

    analyser = AnalysisFactory().create("diagnosis")
    result1 = rc1.get_or_compute("diag_report",
                  lambda df: analyser.run(df), df_c2)
    result2 = rc1.get_or_compute("diag_report",   # should hit cache
                  lambda df: analyser.run(df), df_c2)
    assert result1 == result2
    assert rc1.stats()["hits"] >= 2

    rc1.invalidate("diag_report")
    assert rc1.get("diag_report") is None

    print(f"   Cache stats: {rc1.stats()}")
    print("✅ C3 ReportCache — all tests passed (6/6 marks)")
except Exception as e:
    print(f"❌ C3 failed: {e}")
    import traceback; traceback.print_exc()

❌ C3 failed: 'ReportCache' object has no attribute 'clear'


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\198418498.py", line 30, in <module>
    rc1 = ReportCache(); rc1.clear()
                         ^^^^^^^^^
AttributeError: 'ReportCache' object has no attribute 'clear'


---

## Section D — Advanced Analysis & Transforms (25 marks)

Go beyond basic aggregation. This section tests real intermediate-to-advanced data engineering patterns.

### D1 (7 marks) — Cohort Analysis

### D2 (8 marks) — Cost Intelligence Engine

### D3 (10 marks) — Risk Stratification with NumPy


In [14]:
# ── D1 (7 marks): Cohort Analysis ──────────────────────────────
# Build a CohortAnalyser class (NOT using Factory — standalone):
#
# __init__(df: pd.DataFrame):
#       Stores self.df. Must have transform() columns present.
#
# cohort_by_diagnosis(top_n=5) → pd.DataFrame:
#       For the top_n diagnoses by count, compute per cohort:
#           count, avg_admission_score, avg_discharge_score,
#           avg_score_delta, avg_los_days, readmission_rate,
#           avg_cost_usd, high_risk_rate (% high_risk_flag==True)
#       Return sorted by count descending.
#       Round all floats to 2dp.
#
# cohort_by_ward() → pd.DataFrame:
#       Per ward: count, avg_admission_score, avg_los_days,
#       readmission_rate, avg_cost_usd, pct_critical
#       (% where risk_tier=="Critical"), pct_icu (% where ward=="ICU" — will be 0 or 100)
#       Return sorted by avg_cost_usd descending.
#
# age_band_outcomes() → pd.DataFrame:
#       Per age_group: count, avg_score_delta,
#       readmission_rate, avg_cost_usd, avg_los_days,
#       pct_high_risk (% high_risk_flag)
#       Return sorted by age_group category order.
#
# correlation_matrix(cols=None) → pd.DataFrame:
#       If cols is None, use: ["age","admission_score","discharge_score",
#           "los_days","cost_usd","bmi","hba1c","systolic_bp"]
#       Return df[cols].corr().round(3)
#
# save_all(output_dir="reports") → dict[str, str]:
#       Run all three cohort methods + correlation_matrix.
#       Save each as CSV to output_dir.
#       Return {"cohort_diagnosis": "path", "cohort_ward": "path", ...}

class CohortAnalyser:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    pipe_d = HealthcarePipeline(CSVHealthcareLoader())
    pipe_d.load("healthcare_raw.csv").transform()
    df_d = pipe_d.df.copy()

    ca = CohortAnalyser(df_d)

    diag_cohort = ca.cohort_by_diagnosis(top_n=5)
    assert len(diag_cohort) == 5
    assert "readmission_rate" in diag_cohort.columns
    assert "high_risk_rate"   in diag_cohort.columns
    print(f"   Diagnosis cohorts:\n{diag_cohort[['count','avg_admission_score','readmission_rate']].to_string()}")

    ward_cohort = ca.cohort_by_ward()
    assert "pct_critical" in ward_cohort.columns

    age_outcomes = ca.age_band_outcomes()
    assert "pct_high_risk" in age_outcomes.columns

    corr = ca.correlation_matrix()
    assert corr.shape[0] == corr.shape[1]
    assert corr.loc["admission_score","discharge_score"] != 1.0  # not perfect correlation

    paths = ca.save_all("reports")
    assert all(os.path.exists(p) for p in paths.values())
    print(f"   Saved {len(paths)} cohort reports")
    print("✅ D1 CohortAnalyser — all tests passed (7/7 marks)")
except Exception as e:
    print(f"❌ D1 failed: {e}")
    import traceback; traceback.print_exc()

❌ D1 failed: HealthcarePipeline() takes no arguments


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\3439162186.py", line 43, in <module>
    pipe_d = HealthcarePipeline(CSVHealthcareLoader())
TypeError: HealthcarePipeline() takes no arguments


In [15]:
# ── D2 (8 marks): Cost Intelligence Engine ──────────────────────
# Build CostIntelligence class:
#
# __init__(df: pd.DataFrame, config: HospitalConfig = None):
#       If config is None, use HospitalConfig() (singleton).
#
# outlier_patients(pct=None) → pd.DataFrame:
#       pct defaults to config.get("cost_outlier_pct", 0.95)
#       Return patients with cost_usd > pct quantile.
#       Include: patient_id, name, diagnosis, cost_usd,
#                los_days, cost_per_day, risk_tier
#       Sort by cost_usd descending.
#
# cost_efficiency_score(df_slice=None) → pd.Series:
#       If df_slice is None, use self.df
#       Formula (vectorised, using numpy):
#           efficiency = (score_delta / los_days) / (cost_usd / 1000)
#           Replace inf and nan with 0
#           Clip to range [-10, 10]
#           Round to 3dp
#       Return as pd.Series named "cost_efficiency"
#
# department_roi() → pd.DataFrame:
#       Per department:
#           total_patients, total_cost, avg_cost_per_patient,
#           avg_score_improvement (score_delta mean),
#           avg_efficiency (mean of cost_efficiency_score),
#           readmission_cost_burden: sum of cost_usd where readmitted==True
#       Sort by avg_efficiency descending. Round to 2dp.
#
# savings_opportunity() → dict:
#       Identify cost reduction opportunities:
#       {
#         "high_los_cost": total cost_usd where los_days >
#                          config.get("max_los_days",30),
#         "readmission_burden": total cost_usd of readmitted patients,
#         "outlier_total": sum of outlier_patients() cost_usd,
#         "potential_saving_pct": float — outlier_total / total_cost * 100,
#         "n_outliers": int
#       }
#       All money values rounded to 2dp.
#
# export_cost_report(path="reports/cost_report.csv") → None:
#       Build a comprehensive cost DataFrame with all engineered columns.
#       Include: patient_id, diagnosis, department, ward,
#                cost_usd, cost_per_day, cost_efficiency (computed),
#                risk_tier, readmitted, readmission_cost, los_days
#       Export to path. Print row + col count.

class CostIntelligence:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    HospitalConfig().load_defaults()
    ci = CostIntelligence(df_d)

    outliers = ci.outlier_patients()
    assert len(outliers) > 0
    assert "cost_per_day" in outliers.columns
    print(f"   Outliers (top 5% cost): {len(outliers)} patients")
    print(f"   Top 3:\n{outliers[['patient_id','diagnosis','cost_usd']].head(3).to_string()}")

    eff = ci.cost_efficiency_score()
    assert isinstance(eff, pd.Series)
    assert eff.name == "cost_efficiency"
    assert eff.isna().sum() == 0

    roi = ci.department_roi()
    assert "readmission_cost_burden" in roi.columns
    print(f"\n   Department ROI:\n{roi[['total_patients','avg_efficiency','readmission_cost_burden']].to_string()}")

    savings = ci.savings_opportunity()
    assert "potential_saving_pct" in savings
    print(f"\n   Savings opportunity: ${savings['outlier_total']:,.2f} ({savings['potential_saving_pct']:.1f}%)")

    ci.export_cost_report()
    assert os.path.exists("reports/cost_report.csv")
    print("✅ D2 CostIntelligence — all tests passed (8/8 marks)")
except Exception as e:
    print(f"❌ D2 failed: {e}")
    import traceback; traceback.print_exc()

❌ D2 failed: 'HospitalConfig' object has no attribute 'load_defaults'


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\1189777094.py", line 56, in <module>
    HospitalConfig().load_defaults()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'HospitalConfig' object has no attribute 'load_defaults'


In [16]:
# ── D3 (10 marks): Risk Stratification with NumPy ───────────────
# Build RiskStratifier class — uses NumPy for ALL calculations
# (no pandas .mean()/.std() — use np.nanmean, np.nanstd etc.)
#
# __init__(df: pd.DataFrame):
#       Store df. Pre-compute numpy arrays for speed:
#           self._scores = df["admission_score"].values  (np.ndarray)
#           self._ages   = df["age"].values
#           self._los    = df["los_days"].values
#           self._costs  = df["cost_usd"].values
#
# composite_risk_score() → np.ndarray:
#       Vectorised composite risk formula (all numpy, no loops):
#           Step 1: z_score  = (scores - nanmean) / nanstd
#           Step 2: z_age    = (ages   - nanmean) / nanstd
#           Step 3: z_los    = (los    - nanmean) / nanstd
#           Step 4: composite = 0.50*(-z_score) + 0.30*z_age + 0.20*z_los
#               (inverted score: lower score = higher risk)
#           Step 5: Scale to 0-100:
#               (composite - min) / (max - min) * 100
#           Step 6: Round to 2dp, replace any nan with 50.0
#           Return np.ndarray of same length as df
#
# percentile_bands(scores: np.ndarray=None) → np.ndarray:
#       If scores is None, call composite_risk_score()
#       Assign band label (as string array):
#           >= 80th percentile → "Very High"
#           >= 60th percentile → "High"
#           >= 40th percentile → "Moderate"
#           >= 20th percentile → "Low"
#           else               → "Very Low"
#       Use np.percentile. Return np.ndarray of strings.
#
# flag_anomalies() → pd.DataFrame:
#       Detect statistical anomalies using numpy:
#           cost_anomaly:  |z-score of cost_usd| > 3.0
#           los_anomaly:   los_days > np.percentile(los, 95)
#           score_anomaly: admission_score < np.percentile(scores, 5)
#           bp_anomaly:    systolic_bp > 180 (if column exists)
#       Return df with added boolean columns for each anomaly
#       + "anomaly_count" (sum across 4 flags per row)
#       Filter: return only rows with anomaly_count >= 1
#
# bootstrap_ci(col="admission_score", n=1000,
#              ci=0.95, seed=42) → dict:
#       Bootstrap confidence interval for the column mean.
#       Draw n samples of size len(df) with replacement.
#       Return: {"mean": float, "ci_lower": float, "ci_upper": float,
#                "std_error": float}  all rounded to 3dp
#
# stratified_summary() → pd.DataFrame:
#       Add composite_risk_score and risk_band columns to a copy of df
#       Then groupby risk_band and compute:
#           count, mean_composite_risk (round 2dp),
#           readmission_rate (round 3dp),
#           avg_cost (round 2dp), avg_los (round 2dp)
#       Sort by mean_composite_risk descending

class RiskStratifier:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    rs = RiskStratifier(df_d)

    comp = rs.composite_risk_score()
    assert len(comp) == len(df_d)
    assert comp.min() >= 0 and comp.max() <= 100, f"Range error: {comp.min():.1f}–{comp.max():.1f}"
    assert np.isnan(comp).sum() == 0, "NaNs in composite score"
    print(f"   Composite risk: min={comp.min():.1f} mean={comp.mean():.1f} max={comp.max():.1f}")

    bands = rs.percentile_bands(comp)
    unique_bands = set(bands)
    assert unique_bands == {"Very High","High","Moderate","Low","Very Low"}, f"Unexpected bands: {unique_bands}"
    for b in unique_bands:
        print(f"   {b}: {(bands==b).sum()} patients")

    anomalies = rs.flag_anomalies()
    assert "anomaly_count" in anomalies.columns
    assert len(anomalies) > 0
    print(f"\n   Anomaly patients: {len(anomalies)} ({len(anomalies)/len(df_d)*100:.1f}%)")
    print(f"   Max anomalies per patient: {anomalies['anomaly_count'].max()}")

    bci = rs.bootstrap_ci("admission_score", n=500, seed=42)
    assert "ci_lower" in bci and "ci_upper" in bci
    assert bci["ci_lower"] < bci["mean"] < bci["ci_upper"]
    print(f"\n   Bootstrap CI (admission_score): {bci['ci_lower']} – {bci['ci_upper']}")

    strat = rs.stratified_summary()
    assert "mean_composite_risk" in strat.columns
    print(f"\n   Stratified summary:\n{strat.to_string()}")
    print("✅ D3 RiskStratifier — all tests passed (10/10 marks)")
except Exception as e:
    print(f"❌ D3 failed: {e}")
    import traceback; traceback.print_exc()

❌ D3 failed: name 'df_d' is not defined


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\1921874850.py", line 65, in <module>
    rs = RiskStratifier(df_d)
                        ^^^^
NameError: name 'df_d' is not defined


---

## Section E — Full System Integration (20 marks)

Build the complete end-to-end system that connects every component.

### E1 (10 marks) — `HospitalAnalyticsSystem` — master orchestrator

### E2 (10 marks) — Full pipeline run + final report


In [17]:
# ── E1 (10 marks): HospitalAnalyticsSystem ──────────────────────
# The master orchestrator. Wires everything together.
#
# __init__(data_path: str):
#       Initialise all components:
#           self.config    = HospitalConfig().load_defaults()
#           self.cache     = ReportCache()
#           self.pipeline  = HealthcarePipeline(CSVHealthcareLoader())
#           self.factory   = AnalysisFactory()
#           self._ready    = False
#           self._reports  = {}
#           self.data_path = data_path
#
# load_and_prepare() → self:
#       pipeline.load(data_path).transform()
#       Set self._ready = True. Return self.
#
# run_analysis(use_cache=True) → dict:
#       Guard: RuntimeError if not _ready
#       For each analyser in factory.available():
#           key = f"analysis_{name}"
#           If use_cache: use cache.get_or_compute(key, fn, df)
#           Else: call factory.create(name).run(df) directly
#       Store all in self._reports. Return self._reports.
#
# run_cohort(top_n=5) → dict:
#       Guard: RuntimeError if not _ready
#       Build CohortAnalyser(pipeline.df)
#       Return {
#           "by_diagnosis": cohort_by_diagnosis(top_n).to_dict("records"),
#           "by_ward": cohort_by_ward().to_dict("records"),
#           "age_outcomes": age_band_outcomes().to_dict("records"),
#       }
#
# run_risk(use_cache=True) → dict:
#       Guard: RuntimeError if not _ready
#       If use_cache and "risk_stratification" in cache.keys():
#           return cache.get("risk_stratification")
#       Build RiskStratifier(pipeline.df)
#       result = {
#           "composite_scores": stratified_summary().to_dict("records"),
#           "anomaly_count": len(flag_anomalies()),
#           "bootstrap_ci_admission": bootstrap_ci("admission_score",n=500),
#       }
#       cache.set("risk_stratification", result)
#       return result
#
# run_cost(use_cache=True) → dict:
#       Guard: RuntimeError if not _ready
#       If use_cache and "cost_analysis" in cache.keys():
#           return cache.get("cost_analysis")
#       Build CostIntelligence(pipeline.df, config)
#       result = {
#           "savings_opportunity": savings_opportunity(),
#           "department_roi": department_roi().to_dict("records"),
#           "n_outliers": len(outlier_patients()),
#       }
#       cache.set("cost_analysis", result)
#       return result
#
# full_report(output_dir="reports") → dict:
#       Run: run_analysis(), run_cohort(), run_risk(), run_cost()
#       Save pipeline.df to f"{output_dir}/final_dataset.csv"
#       CohortAnalyser(pipeline.df).save_all(output_dir)
#       CostIntelligence(pipeline.df).export_cost_report(f"{output_dir}/cost_report.csv")
#       Return combined summary dict:
#       {
#         "hospital": config.get("hospital_name"),
#         "fiscal_year": config.get("fiscal_year"),
#         "total_patients": len(pipeline.df),
#         "analyses_run": list of analyser names,
#         "cache_stats": cache.stats(),
#         "savings_opportunity": from run_cost(),
#         "anomaly_count": from run_risk(),
#         "files_exported": list of output file paths,
#       }
#
# __repr__: show hospital name, ready status, patient count if ready

class HospitalAnalyticsSystem:
    # YOUR CODE HERE
    pass

# ── Tests ────────────────────────────────────────────────────────
try:
    # Reset singletons for clean test
    HospitalConfig().reset()
    ReportCache().clear()

    system = HospitalAnalyticsSystem("healthcare_raw.csv")
    system.load_and_prepare()
    assert system._ready

    analyses = system.run_analysis(use_cache=True)
    assert len(analyses) == 3
    print(f"   Analyses: {list(analyses.keys())}")

    cohort = system.run_cohort(top_n=4)
    assert "by_diagnosis" in cohort and len(cohort["by_diagnosis"]) == 4

    risk = system.run_risk(use_cache=True)
    assert "anomaly_count" in risk and "bootstrap_ci_admission" in risk
    print(f"   Risk: {risk['anomaly_count']} anomalies detected")

    cost = system.run_cost(use_cache=True)
    assert "savings_opportunity" in cost
    print(f"   Cost: ${cost['savings_opportunity']['potential_saving_pct']:.1f}% outlier burden")

    # Second call should hit cache
    _ = system.run_risk(use_cache=True)
    _ = system.run_cost(use_cache=True)
    stats = ReportCache().stats()
    assert stats["hits"] >= 2, f"Cache not being used: {stats}"
    print(f"   Cache stats: {stats}")

    print("✅ E1 HospitalAnalyticsSystem — all tests passed (10/10 marks)")
except Exception as e:
    print(f"❌ E1 failed: {e}")
    import traceback; traceback.print_exc()

❌ E1 failed: 'HospitalConfig' object has no attribute 'reset'


Traceback (most recent call last):
  File "C:\Users\deepak347401\AppData\Local\Temp\1\ipykernel_7108\914155388.py", line 87, in <module>
    HospitalConfig().reset()
    ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'HospitalConfig' object has no attribute 'reset'


In [18]:
# ── E2 (10 marks): Full Pipeline Run + Final Report ─────────────
# This cell ties everything together and produces the final deliverable.
# No skeleton — you write the entire orchestration.
#
# Tasks:
# 1. Initialise the system with config overrides:
#       score_threshold = 62.0
#       high_risk_threshold = 38.0
#       hospital_name = "EXL General Hospital — Batch 2026"
#
# 2. Load, transform, run all analyses
#
# 3. Use the AnalysisFactory to run ALL analysers and print
#    a formatted summary of each report (key → value, max 3 rows each)
#
# 4. Run CohortAnalyser and print the top-5 diagnosis cohort table
#    in a clean, readable format
#
# 5. Run RiskStratifier and print:
#    - Stratified summary table
#    - Bootstrap CI for admission_score
#    - How many patients have 3+ anomalies
#
# 6. Run CostIntelligence and print:
#    - Department ROI table (sorted by avg_efficiency)
#    - Savings opportunity summary
#    - Top 5 cost outliers
#
# 7. Generate full_report() — print the summary dict
#
# 8. List all files in the reports/ directory and their sizes
#
# 9. Final assertion: all of the following files must exist:
#    reports/final_dataset.csv
#    reports/cost_report.csv
#    reports/cohort_diagnosis.csv
#    reports/cohort_ward.csv
#
# 10. Print a formatted banner:
#     ╔══════════════════════════════════════════════╗
#     ║   EXL Healthcare Analytics — Run Complete   ║
#     ║   Patients: XXX | Analyses: 3 | Files: X    ║
#     ╚══════════════════════════════════════════════╝

# YOUR CODE HERE — no skeleton provided



---

## 📝 Reflection (Not graded — but required for submission)

Answer each question in 2–4 sentences. Replace the placeholder text.


In [19]:
reflection = {
    "Q1_factory_vs_if_elif": (
        "YOUR ANSWER: Explain one real scenario in the pipeline above where replacing "
        "if/elif with Factory Pattern made your code easier to extend."
    ),
    "Q2_singleton_tradeoff": (
        "YOUR ANSWER: What is one disadvantage of the Singleton pattern you noticed "
        "while building HospitalConfig? When would you NOT use it?"
    ),
    "Q3_abc_value": (
        "YOUR ANSWER: Give one concrete example from this assignment where the ABC "
        "contract caught a mistake earlier than it would have without it."
    ),
    "Q4_numpy_vs_pandas": (
        "YOUR ANSWER: In D3 RiskStratifier you used numpy arrays directly. "
        "When is this faster than using pandas, and when is it not worth it?"
    ),
    "Q5_what_would_you_add": (
        "YOUR ANSWER: If you had 2 extra hours, which single feature would you add "
        "to HospitalAnalyticsSystem and why?"
    ),
}

for q, a in reflection.items():
    print(f"\n{q}:")
    print(f"  {a}")


Q1_factory_vs_if_elif:
  YOUR ANSWER: Explain one real scenario in the pipeline above where replacing if/elif with Factory Pattern made your code easier to extend.

Q2_singleton_tradeoff:
  YOUR ANSWER: What is one disadvantage of the Singleton pattern you noticed while building HospitalConfig? When would you NOT use it?

Q3_abc_value:
  YOUR ANSWER: Give one concrete example from this assignment where the ABC contract caught a mistake earlier than it would have without it.

Q4_numpy_vs_pandas:
  YOUR ANSWER: In D3 RiskStratifier you used numpy arrays directly. When is this faster than using pandas, and when is it not worth it?

Q5_what_would_you_add:
  YOUR ANSWER: If you had 2 extra hours, which single feature would you add to HospitalAnalyticsSystem and why?


---

## ✅ Submission Checklist

Run this cell last. All checks must pass before you push to GitHub.


In [20]:
print("=" * 60)
print("SUBMISSION CHECKLIST")
print("=" * 60)

checks = []

# File checks
required_files = [
    "reports/final_dataset.csv",
    "reports/cost_report.csv",
    "reports/cohort_diagnosis.csv",
    "reports/cohort_ward.csv",
]
for f in required_files:
    exists = os.path.exists(f)
    checks.append((f"File exists: {f}", exists))

# Class checks
for cls_name in ["Vitals","PatientRecord","HospitalBed",
                 "HealthcareDataLoader","CSVHealthcareLoader","HealthcarePipeline",
                 "HealthcareAnalyser","DiagnosisAnalyser","DemographicsAnalyser",
                 "ClinicalRiskAnalyser","AnalysisFactory",
                 "HospitalConfig","ReportCache",
                 "CohortAnalyser","CostIntelligence","RiskStratifier",
                 "HospitalAnalyticsSystem"]:
    defined = cls_name in dir()
    checks.append((f"Class defined: {cls_name}", defined))

# Singleton checks
try:
    c1, c2 = HospitalConfig(), HospitalConfig()
    checks.append(("HospitalConfig is singleton", c1 is c2))
    r1, r2 = ReportCache(), ReportCache()
    checks.append(("ReportCache is singleton", r1 is r2))
except: checks.append(("Singleton checks", False))

# Reflection check
try:
    incomplete = sum(1 for v in reflection.values() if "YOUR ANSWER" in v)
    checks.append(("Reflection complete", incomplete == 0))
except: checks.append(("Reflection defined", False))

passed = sum(1 for _, ok in checks if ok)
total  = len(checks)

for label, ok in checks:
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")

print()
print(f"Score: {passed}/{total} checks passed")
if passed == total:
    print()
    print("🎉 All checks passed! Ready to submit.")
    print()
    print("Run these commands:")
    print("  git add day3_assignment.ipynb reports/")
    print("  git commit -m 'Day3 assignment complete'")
    print("  git push")
else:
    missing = total - passed
    print(f"⚠️  {missing} check(s) failing. Fix before submitting.")

SUBMISSION CHECKLIST
  ❌  File exists: reports/final_dataset.csv
  ❌  File exists: reports/cost_report.csv
  ❌  File exists: reports/cohort_diagnosis.csv
  ❌  File exists: reports/cohort_ward.csv
  ✅  Class defined: Vitals
  ✅  Class defined: PatientRecord
  ✅  Class defined: HospitalBed
  ✅  Class defined: HealthcareDataLoader
  ✅  Class defined: CSVHealthcareLoader
  ✅  Class defined: HealthcarePipeline
  ❌  Class defined: HealthcareAnalyser
  ❌  Class defined: DiagnosisAnalyser
  ❌  Class defined: DemographicsAnalyser
  ❌  Class defined: ClinicalRiskAnalyser
  ❌  Class defined: AnalysisFactory
  ✅  Class defined: HospitalConfig
  ✅  Class defined: ReportCache
  ✅  Class defined: CohortAnalyser
  ✅  Class defined: CostIntelligence
  ✅  Class defined: RiskStratifier
  ✅  Class defined: HospitalAnalyticsSystem
  ❌  HospitalConfig is singleton
  ❌  ReportCache is singleton
  ❌  Reflection complete

Score: 12/24 checks passed
⚠️  12 check(s) failing. Fix before submitting.
